# **Use Python to match U.S. Census population and housing permit data with Wharton data set**

# Goal: Build a data set the shows whether places with stricter zoning less building

This notebook builds a data set with towns as rows and colums for:
1. **zoning strictness**, from the Wharton Residential Land Use Regulatory Index 2018
**Factors weighed in score:**
- LPPI18 => "Local Political Pressure Index, 2018"
- SPII18 => "State Political Involvement Index, 2018"
- CII18 => "Court Involvement Index, 2018"
- LPAI18 => "Local Project Approval Index, 2018"
- LZAI18 => "Local Zoning Approval Index, 2018"
- LAI18 => "Local Assembly Index, 2018"
- SRI18 => "Supply Restrictions Index, 2018"
- DRI18 => "Density Restriction Index, 2018"
- OSI18 => "Open Space Index, 2018"
- EI18 => "Exactions Index, 2018"
- AHI18 => "Affordable Housing Index, 2018"
- ADI18 => "Approval Delay Index, 2018"

2. **building permits**, 2018–2022, from the Census Building Permits Survey
- I chose these years becuase they included 2018 when the Wharton survey came out, but also including
other years to protect from the effects of outliers.

3. **population**, from Census population estimates (so permits can be per resident)
- I loaded 2010-2017 so I could measure the growth occuring in these places before the Wharton survey.

**How I matched the data sets:** I used the Census location code instead of the town names to avoid differences in spelling and format.

Data will be stored in `analysis/wharton_permits_matched.csv`

## Setup

In [19]:
import pandas as pd

## Step 1: Load the Wharton survey
https://real-estate.wharton.upenn.edu/profile/gyourko/ --> Make a copy from the dta. file to a csv

Wharton's `GEOID` column holds the same location code that's used in census data:
- **7 digits** = a place (state + place code) THIS IS THE WAY I WANT IT TO BE FORMATTED
- **10 digits** = a county subdivision (state + county + subdivision code). We keep state + the last 5 digits so it matches the permit file. DROP THE THREE COUNTRY DIGITS
- **5 digits** = a county government. I dropped these beacuse they have no matching city permits.

In [20]:
# Open the Wharton data and load only the columns needed
wh_raw = pd.read_csv("../raw_data/wharton/WRLURI_01_15_2020.csv",
                     usecols=["GEOID", "communityname18", "state", "countyname18", "cbsatitle18",
                              "WRLURI18", "ADI18", "q16b118", "q17a118", "DRI18", "SRI18", "LZAI18", "LPPI18",
                              "SPII18", "CII18", "LPAI18", "LAI18", "OSI18", "EI18", "AHI18"])
print("Wharton rows:", len(wh_raw))

# Read every GEOID as text not a number and stor it without anything after the '.0'
wh_raw["geoid"] = wh_raw["GEOID"].astype(str).apply(lambda x: x.split(".")[0])
#Count towns have 7,10 and 5 digits and store it in a table called number of digits
wh_raw["geoid"].apply(len).value_counts().rename("number of digits")

Wharton rows: 2844


geoid
7     1938
10     894
5       12
Name: number of digits, dtype: int64

In [21]:
# Label each row and build a 7-digit match key
def label_type(geoid):
    if len(geoid) == 7: #7-digits = city/village/borough
        return "place"
    if len(geoid) == 10: #10-digits = townships (state (first 2 digits) + county(middle 3 digits) + township (last 5 digits))
        return "cousub"
    return "county" #5-digits or UNINC = county

def make_match_key(geoid):
    if len(geoid) == 7:
        return geoid #Keep as is. Ideal format!
    if len(geoid) == 10:
        return geoid[:2] + geoid[-5:] # Drop the 3 county digits in the middle
    return None # Mark counties as blank

wh_raw["geo_type"] = wh_raw["geoid"].apply(label_type)
wh_raw["match_key"] = wh_raw["geoid"].apply(make_match_key)

# Drop county governments because they don't match the 7-digit town code the data sets will be matched with
wh = wh_raw[wh_raw["geo_type"] != "county"]
print("Wharton towns kept:", len(wh))
print(wh["geo_type"].value_counts())

#Drop the places without a WRLURI18 score.
wh = wh.dropna(subset=["WRLURI18"])

Wharton towns kept: 2832
geo_type
place     1938
cousub     894
Name: count, dtype: int64


## Step 2: Load the Census building permits, 2018–2022
https://www2.census.gov/econ/bps/Place/ 

The Census Building Permits Survey publishes text files of how many housing permits every permit-issuing place issued
Important to note:
1. **Includes number of units per building:** 1 unit, 2 units, 3–4 units, 5+ units
2. **Published every year**
3. **Divided by region:** Midwest, Northeast, South, West
4. **"Months reported" = 0:** Means the town didn't submit data that year, not that it was actually 0.

In [22]:
# Retrieve all 20 regional permitting data sets from Census
regions = ["mw", "ne", "so", "we"]    # Midwest, Northeast, South, West
years = [2018, 2019, 2020, 2021, 2022]

files = []
for region in regions:
    for year in years:
        path = f"../raw_data/census/bps/{region}{year}a.txt"
        files.append(pd.read_csv(path, skiprows=3, header=None, dtype=str, encoding="latin-1")) #the file has two header lines and one blank line at the top this code makes sure these are skipped so it can run. It also prompts pandas to make numbered columns and turns everything into text.

# Combine 20 data sets into a new data frame, clean it and print its size
bps = pd.concat(files, ignore_index=True)
bps = bps.apply(lambda col: col.str.strip()) #Removes padding spaces at the beginning and end of town names.
print(bps.shape)

(100429, 41)


In [23]:
# Keep only these columns (the Census file gives a key for number meanings)
bps = bps[[0, 1, 5, 6, 15, 16, 18, 21, 24, 27]]

# Make columns have simpler names
bps = bps.rename(columns={
    0: "year",
    1: "state_code",
    5: "place_code",
    6: "cousub_code",
    15: "months_reported",
    16: "bps_name",
    18: "units_1",
    21: "units_2",
    24: "units_3_4",
    27: "units_5plus",
})

num_cols = ["months_reported", "units_1", "units_2", "units_3_4", "units_5plus"]
bps[num_cols] = bps[num_cols].astype(int)

In [24]:
# Create columns for total new homes and apartments
# (Note: apartments are classified as any building with 2 or more residential units)
bps["units_all"] = bps["units_1"] + bps["units_2"] + bps["units_3_4"] + bps["units_5plus"]
bps["units_apartments"] = bps["units_all"] - bps["units_1"]

# Remove all rows where munipalities didn't submit data
# 0 means missing data not 0 homes built
print(f"share of rows not reported: {(bps['months_reported'] == 0).mean():.3f}") #Asked Claude for help with this f string
bps = bps[bps["months_reported"] > 0]
bps.head()

share of rows not reported: 0.189


,year,state_code,place_code,cousub_code,months_reported,bps_name,units_1,units_2,units_3_4,units_5plus,units_all,units_apartments
0,2018,17,00113,11891,12,Abingdon,0,0,0,0,0,0
1,2018,17,00243,00250,12,Addison village,30,0,0,0,30,0
2,2018,17,00516,00529,12,Albany village,0,0,0,0,0,0
3,2018,17,00555,44667,12,Albers village,0,0,0,0,0,0
4,2018,17,00646,48398,12,Aledo,0,0,0,0,0,0


## Step 3: Match permits to Wharton towns by location code

**The Permit data set does not have the same 7-digit code so I had to combine the state code with the local code.**
There are two types of places recorded:
- **Cities:** `place_code`
- **Towns/townships:** `cousub_code`

In [25]:
# Label each town in the permit data with the same 7-digit code used in the Wharton data (they will later be matched)
#Cities and Townships must be kept in separate lists in case two hav ethe same 5-digit code
place_keys = wh[wh["geo_type"] == "place"]["match_key"]
cousub_keys = wh[wh["geo_type"] == "cousub"]["match_key"]

place_key = bps["state_code"] + bps["place_code"]
cousub_key = bps["state_code"] + bps["cousub_code"]

#Since there are villages inside of townships, do not match villages with the same code as townships
bps["match_key"] = None
bps.loc[place_key.isin(place_keys), "match_key"] = place_key
bps.loc[cousub_key.isin(cousub_keys) & (bps["place_code"] == "00000"), "match_key"] = cousub_key

#This removes any unmatched places
matched = bps.dropna(subset=["match_key"])
print("permit rows matched to a Wharton town:", len(matched))

permit rows matched to a Wharton town: 9710


In [26]:
# Add up each town's permits over all years, and count how many years it reported
permits = matched.groupby("match_key").agg({
    "units_1": "sum",
    "units_apartments": "sum",
    "units_all": "sum",
    "year": "count",
}).reset_index().rename(columns={"year": "years_reported"})

# Divide the totals by the number of years with data to get the average per year
permits["units_1_avg_2018_22"] = permits["units_1"] / permits["years_reported"]
permits["units_apartments_avg_2018_22"] = permits["units_apartments"] / permits["years_reported"]
permits["units_all_avg_2018_22"] = permits["units_all"] / permits["years_reported"]

# Add the town's name from the permit file (used to check the match in Step 5)
names = matched.groupby("match_key").agg({"bps_name": "first"}).reset_index()
permits = pd.merge(permits, names, on="match_key")

# Exclude towns that don't have at least 3 years of permit data to avoid outliers
permits = permits[permits["years_reported"] >= 3]
print("Wharton towns with 3+ years of permits:", len(permits))
permits.head()


Wharton towns with 3+ years of permits: 1995


,match_key,units_1,units_apartments,units_all,years_reported,units_1_avg_2018_22,units_apartments_avg_2018_22,units_all_avg_2018_22,bps_name
0,0103076,2969,2885,5854,5,593.8,577.0,1170.8,Auburn
2,0115136,225,56,281,5,45.0,11.2,56.2,Clanton
3,0119648,1833,6,1839,5,366.6,1.2,367.8,Daphne
4,0120296,3,0,3,3,1.0,0.0,1.0,Demopolis
5,0121184,1060,121,1181,5,212.0,24.2,236.2,Dothan


## Step 4: Population
https://www2.census.gov/programs-surveys/popest/datasets/

**This data requires the same procedure as the permit data to get the 7-digit location code**
Download the data sets needs for 2018-2022 and to measure growth before and after
- 2010s file: 2010 census count, 2017–2019 estimates
- 2020s file: 2020–2023 estimates

In [27]:
# Load the two population files (downloaded from Census into data/census/)
pop_2010s = pd.read_csv("../raw_data/census/sub-est2019_all.csv", encoding="latin-1", dtype=str)
pop_2020s = pd.read_csv("../raw_data/census/sub-est2023.csv", encoding="latin-1", dtype=str)

In [28]:
# Create the 7-digit code to match to the other data sets by combining the town/city and state codes
# Cities/places= state + place code
places = pop_2010s[pop_2010s["SUMLEV"] == "162"]
places["match_key"] = places["STATE"] + places["PLACE"]

# Townships = state + township code
cousubs = pop_2010s[pop_2010s["SUMLEV"] == "061"]
cousubs["match_key"] = cousubs["STATE"] + cousubs["COUSUB"]

# Create a table for the cities and townships with their new codes
pop_2010s = pd.concat([places, cousubs])

In [29]:
# Remove duplicates (same 7-digit code listed twice)
pop_2010s = pop_2010s.groupby("match_key")[["CENSUS2010POP", "POPESTIMATE2017",
                                            "POPESTIMATE2018", "POPESTIMATE2019"]].first()

# Don't count cells that list A instead of a number
pop_2010s = pop_2010s.replace("A", None).astype(float).reset_index()
pop_2010s.head()

,match_key,CENSUS2010POP,POPESTIMATE2017,POPESTIMATE2018,POPESTIMATE2019
0,0100124,2688.0,2575.0,2571.0,2560.0
1,0100460,4522.0,4327.0,4308.0,4281.0
2,0100484,758.0,726.0,723.0,718.0
3,0100676,356.0,332.0,331.0,328.0
4,0100820,30352.0,33275.0,33413.0,33487.0


In [30]:
# Same steps for the 2020s file
places = pop_2020s[pop_2020s["SUMLEV"] == "162"]
places["match_key"] = places["STATE"] + places["PLACE"]

cousubs = pop_2020s[pop_2020s["SUMLEV"] == "061"]
cousubs["match_key"] = cousubs["STATE"] + cousubs["COUSUB"]

pop_2020s = pd.concat([places, cousubs])

In [31]:
# Remove duplicates (same 7-digit code listed twice) and keep only the population columns needed
pop_2020s = pop_2020s.groupby("match_key")[["POPESTIMATE2020", "POPESTIMATE2021",
                                            "POPESTIMATE2022", "POPESTIMATE2023"]].first()

# Don't count cells that list A instead of a number
pop_2020s = pop_2020s.replace("A", None).astype(float).reset_index()
pop_2020s.head()

,match_key,POPESTIMATE2020,POPESTIMATE2021,POPESTIMATE2022,POPESTIMATE2023
0,0100124,2351.0,2357.0,2364.0,2377.0
1,0100460,4360.0,4292.0,4222.0,4172.0
2,0100484,660.0,666.0,670.0,667.0
3,0100676,229.0,228.0,223.0,224.0
4,0100820,33392.0,33748.0,33869.0,34107.0


In [32]:
#Combine the two population data into one
pop_all = pd.merge(pop_2010s, pop_2020s, on="match_key", how="outer")

# Keep only 2010, 2017, 2020, 2022, 2023 as separate columns.
population = pop_all[["match_key", "CENSUS2010POP", "POPESTIMATE2017", "POPESTIMATE2020",
                      "POPESTIMATE2022", "POPESTIMATE2023"]].rename(columns={
    "CENSUS2010POP": "pop_2010",
    "POPESTIMATE2017": "pop_2017",
    "POPESTIMATE2020": "pop_2020",
    "POPESTIMATE2022": "pop_2022",
    "POPESTIMATE2023": "pop_2023",
})

# Create a new data frame with rows for towns and columns for population
population["pop_avg_2018_22"] = pop_all[["POPESTIMATE2018", "POPESTIMATE2019", "POPESTIMATE2020",
                                         "POPESTIMATE2021", "POPESTIMATE2022"]].mean(axis=1)
population.head()

,match_key,pop_2010,pop_2017,pop_2020,pop_2022,pop_2023,pop_avg_2018_22
0,0100124,2688.0,2575.0,2351.0,2364.0,2377.0,2440.6
1,0100460,4522.0,4327.0,4360.0,4222.0,4172.0,4292.6
2,0100484,758.0,726.0,660.0,670.0,667.0,687.4
3,0100676,356.0,332.0,229.0,223.0,224.0,267.8
4,0100820,30352.0,33275.0,33392.0,33869.0,34107.0,33581.8


## Step 5: Time to actually combine everything!!!!

In [33]:
# Match Wharton data, permits data and population data based on the 7-digit code
towns = pd.merge(wh, permits, on="match_key", how="left")
towns = pd.merge(towns, population, on="match_key", how="left")

# Count how many Wharton towns have permit data, population data or both
print("Wharton towns:                  ", len(towns))
print("  with permits (3+ years):      ", towns["units_all_avg_2018_22"].count())
print("  with population:              ", towns["pop_avg_2018_22"].count())
print("  with both:                    ", len(towns.dropna(subset=["units_all_avg_2018_22", "pop_avg_2018_22"])))

Wharton towns:                   2462
  with permits (3+ years):       1995
  with population:               2461
  with both:                     1995


In [34]:
#Claude helped me a lot with this!!!!
# Keep only the towns from the Wharton data that were matched with permit data
both = towns.dropna(subset=["bps_name"])
# Check the code matching worked: the town name in Wharton and in the permit data should agree
# Clean names for small spelling differences like "Saint" vs "St.",
def clean(name): #Learned this from Claude
    name = name.lower().replace("saint ", "st ")
    letters = ""
    for ch in name:
        if ch in "abcdefghijklmnopqrstuvwxyz ":
            letters = letters + ch
        else:
            letters = letters + " "
    return letters

both["names_agree"] = both.apply(lambda row: clean(row["communityname18"]).split()[0] in clean(row["bps_name"]), axis=1)
print(f"names agree for {both['names_agree'].mean():.1%} of matched towns")
# Show a list of the towns with the same code but different names
both[both["names_agree"] == False][["communityname18", "state", "bps_name"]].head(15)

names agree for 99.8% of matched towns


,communityname18,state,bps_name
1846,Woodard,PA,Woodward township
1959,Cummings,GA,Cumming
2441,Williamston,MI,Williamstown township


In [35]:
# Show list of towns that didn't have matches
# This is likely because they don't issue their own permits or have data for less than 3 years
no_permits = towns[towns["units_all_avg_2018_22"].isna()]
no_permits[["communityname18", "state", "geo_type"]].head(10)

,communityname18,state,geo_type
0,Southwick,MA,cousub
12,Milford,NH,cousub
22,Rockland,ME,place
25,Westminster,VT,cousub
28,Newport,VT,place
30,Plainville,CT,cousub
62,Seaford,DE,place
66,Strasburg,VA,place
67,Rocky Mount,VA,place
73,Summersville,WV,place


In [36]:
#Create column for permits granted per 1,000 based on census population data.
towns["permits_per_thousand"]= (towns["units_all_avg_2018_22"] / towns["pop_avg_2018_22"]) * 1000

towns

,GEOID,state,countyname18,communityname18,cbsatitle18,LPPI18,SPII18,CII18,LPAI18,LZAI18,...,units_apartments_avg_2018_22,units_all_avg_2018_22,bps_name,pop_2010,pop_2017,pop_2020,pop_2022,pop_2023,pop_avg_2018_22,permits_per_thousand
0,2501365825,MA,Hampden County,Southwick,"Springfield, MA",8.0,2.0,4.0,5.0,5.0,...,NaN,NaN,NaN,9502.0,9719.0,9220.0,9185.0,9195.0,9424.6,NaN
1,2502701885,MA,Worcester County,Ashburnham,"Worcester, MA-CT",8.0,2.0,4.0,3.0,3.0,...,0.000000,20.400000,Ashburnham town,6081.0,6308.0,6312.0,6385.0,6415.0,6350.2,3.212497
2,2501702130,MA,Middlesex County,Ashland,"Boston-Cambridge-Newton, MA-NH",11.0,2.0,2.0,3.0,2.0,...,12.800000,28.200000,Ashland town,16593.0,17784.0,18835.0,18572.0,18591.0,18333.0,1.538210
3,2502706365,MA,Worcester County,Bolton,"Worcester, MA-CT",10.0,2.0,2.0,5.0,5.0,...,0.666667,20.333333,Bolton town,4897.0,5283.0,5668.0,5737.0,5738.0,5578.2,3.645142
4,2501768050,MA,Middlesex County,Stow,"Boston-Cambridge-Newton, MA-NH",11.0,1.0,4.0,5.0,2.0,...,0.000000,9.200000,Stow town,6590.0,7083.0,7158.0,7081.0,7088.0,7150.2,1.286677
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2457,1772052,IL,McHenry County,Spring Grove,"Chicago-Naperville-Elgin, IL-IN-WI",7.0,1.0,2.0,4.0,2.0,...,0.000000,9.200000,Spring Grove village,5778.0,5708.0,5480.0,5482.0,5512.0,5579.4,1.648923
2458,5578375,WI,Waukesha County,Summit,"Milwaukee-Waukesha-West Allis, WI",7.0,2.0,2.0,3.0,4.0,...,0.000000,82.200000,Summit village,NaN,4915.0,4824.0,5138.0,5287.0,4978.0,16.512656
2459,1778227,IL,Lake County,Volo,"Chicago-Naperville-Elgin, IL-IN-WI",10.0,1.0,2.0,1.0,3.0,...,45.800000,98.800000,Volo village,2929.0,5465.0,6209.0,6584.0,7016.0,6098.2,16.201502
2460,1782855,IL,McHenry County,Wonder Lake,"Chicago-Naperville-Elgin, IL-IN-WI",11.0,3.0,4.0,3.0,6.0,...,0.400000,82.400000,Wonder Lake village,4026.0,3905.0,3966.0,4456.0,4889.0,4033.8,20.427389


In [37]:
#Drop the rows that don't have permits per capita

towns = towns.dropna(subset=["permits_per_thousand"])
towns

,GEOID,state,countyname18,communityname18,cbsatitle18,LPPI18,SPII18,CII18,LPAI18,LZAI18,...,units_apartments_avg_2018_22,units_all_avg_2018_22,bps_name,pop_2010,pop_2017,pop_2020,pop_2022,pop_2023,pop_avg_2018_22,permits_per_thousand
1,2502701885,MA,Worcester County,Ashburnham,"Worcester, MA-CT",8.0,2.0,4.0,3.0,3.0,...,0.000000,20.400000,Ashburnham town,6081.0,6308.0,6312.0,6385.0,6415.0,6350.2,3.212497
2,2501702130,MA,Middlesex County,Ashland,"Boston-Cambridge-Newton, MA-NH",11.0,2.0,2.0,3.0,2.0,...,12.800000,28.200000,Ashland town,16593.0,17784.0,18835.0,18572.0,18591.0,18333.0,1.538210
3,2502706365,MA,Worcester County,Bolton,"Worcester, MA-CT",10.0,2.0,2.0,5.0,5.0,...,0.666667,20.333333,Bolton town,4897.0,5283.0,5668.0,5737.0,5738.0,5578.2,3.645142
4,2501768050,MA,Middlesex County,Stow,"Boston-Cambridge-Newton, MA-NH",11.0,1.0,4.0,5.0,2.0,...,0.000000,9.200000,Stow town,6590.0,7083.0,7158.0,7081.0,7088.0,7150.2,1.286677
5,2500538225,MA,Bristol County,Mansfield,"Providence-Warwick, RI-MA",9.0,2.0,3.0,3.0,3.0,...,2.400000,11.600000,Mansfield town,23184.0,23931.0,23862.0,23827.0,23862.0,24027.6,0.482778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2457,1772052,IL,McHenry County,Spring Grove,"Chicago-Naperville-Elgin, IL-IN-WI",7.0,1.0,2.0,4.0,2.0,...,0.000000,9.200000,Spring Grove village,5778.0,5708.0,5480.0,5482.0,5512.0,5579.4,1.648923
2458,5578375,WI,Waukesha County,Summit,"Milwaukee-Waukesha-West Allis, WI",7.0,2.0,2.0,3.0,4.0,...,0.000000,82.200000,Summit village,NaN,4915.0,4824.0,5138.0,5287.0,4978.0,16.512656
2459,1778227,IL,Lake County,Volo,"Chicago-Naperville-Elgin, IL-IN-WI",10.0,1.0,2.0,1.0,3.0,...,45.800000,98.800000,Volo village,2929.0,5465.0,6209.0,6584.0,7016.0,6098.2,16.201502
2460,1782855,IL,McHenry County,Wonder Lake,"Chicago-Naperville-Elgin, IL-IN-WI",11.0,3.0,4.0,3.0,6.0,...,0.400000,82.400000,Wonder Lake village,4026.0,3905.0,3966.0,4456.0,4889.0,4033.8,20.427389


## Step 6: Save

In [38]:
towns.to_csv("../analysis/wharton_permits_matched.csv", index=False)
print("saved", len(towns), "towns")

saved 1995 towns
